# Lección 05 — Memoria y RAG Agéntico con Claude

En este notebook vas a construir un agente que puede consultar **tu propia base de conocimiento** antes de responder.

### Lo que vas a aprender:
1. Crear una base de conocimiento propia (simulando documentos reales)
2. Construir un agente RAG que busca antes de responder
3. Implementar el patrón Maker-Checker para respuestas más precisas
4. Agregar memoria de usuario que persiste entre consultas

In [ ]:
%pip install anthropic python-dotenv -q

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
print("Setup listo.")

## Parte 1 — La Base de Conocimiento

En un proyecto real, tu base de conocimiento sería un archivo PDF, una base de datos o documentos de tu negocio.

Acá la simulamos con un diccionario Python, pero el concepto es exactamente el mismo.
En producción reemplazarías `CONOCIMIENTO` por una búsqueda real en tus documentos.

In [ ]:
# Base de conocimiento — simula documentos de tu negocio/curso
CONOCIMIENTO = {
    "claude_code": """Claude Code es la CLI oficial de Anthropic para usar Claude desde la terminal.
    Permite hacer tareas de programación, análisis de código y automatización directamente desde el terminal.
    Se instala con: npm install -g @anthropic-ai/claude-code
    Requiere una API key de Anthropic configurada como variable de entorno.
    Precio: incluido en la suscripción Claude Pro o según uso de API.""",
    
    "claude_api": """La API de Claude (Anthropic API) permite integrar Claude en cualquier aplicación.
    Se accede desde console.anthropic.com donde podés crear API keys.
    Modelos disponibles: claude-opus-4-5 (más potente), claude-sonnet-4-5 (balance), claude-haiku-4-5 (más rápido).
    Precio: basado en tokens — input y output se cobran por separado.
    Librería Python: pip install anthropic""",
    
    "mcp": """MCP (Model Context Protocol) es un protocolo abierto creado por Anthropic.
    Permite conectar Claude con herramientas externas: bases de datos, APIs, sistemas de archivos, etc.
    Los servidores MCP se pueden instalar y configurar en Claude Desktop o Claude Code.
    Ejemplos de servidores MCP disponibles: filesystem, github, google-drive, slack, postgres.
    Es el corazón de la arquitectura agéntica de Claude.""",
    
    "agentes": """Un agente de IA es un sistema que usa un LLM para razonar y tomar acciones autónomas.
    Los agentes pueden usar herramientas (tools) para interactuar con el mundo real.
    El bucle agente funciona así: recibir input → razonar → actuar → observar → repetir.
    Los patrones principales son: tool use, planificación, multi-agente y RAG agéntico.
    Claude es especialmente bueno en razonamiento multi-paso y uso de herramientas.""",
    
    "comunidad": """La comunidad Pol Desaf-IA es un curso de agentes de IA con Claude en español.
    Está dirigida a personas sin experiencia previa en programación de Latinoamérica y España.
    Las lecciones cubren: fundamentos de agentes, herramientas, patrones de diseño, RAG y producción.
    El curso usa Python y la API de Anthropic como herramientas principales.
    Acceso: comunidad en Skool con recursos descargables y soporte directo.""",
    
    "python_basico": """Para seguir este curso necesitás conocimientos básicos de Python.
    Variables, funciones, listas y diccionarios son los conceptos clave.
    No necesitás saber programar profesionalmente — con lo básico alcanza.
    Recursos recomendados: Python para todos (py4e.com), Automate the Boring Stuff.
    Tiempo estimado para aprender lo necesario: 2-4 semanas con práctica diaria."""
}

print(f"Base de conocimiento con {len(CONOCIMIENTO)} documentos cargada.")
print(f"Temas disponibles: {list(CONOCIMIENTO.keys())}")

In [ ]:
# Función de búsqueda en la base de conocimiento
def buscar_conocimiento(consulta: str) -> str:
    """Busca información relevante en la base de conocimiento."""
    consulta_lower = consulta.lower()
    resultados = []
    
    for tema, contenido in CONOCIMIENTO.items():
        # Búsqueda simple por palabras clave
        palabras_clave = consulta_lower.split()
        coincidencias = sum(1 for palabra in palabras_clave if palabra in contenido.lower() or palabra in tema.lower())
        if coincidencias > 0:
            resultados.append((coincidencias, tema, contenido))
    
    if not resultados:
        return "No encontré información relevante sobre ese tema en la base de conocimiento."
    
    # Ordenar por relevancia y devolver los mejores resultados
    resultados.sort(reverse=True)
    salida = []
    for _, tema, contenido in resultados[:2]:  # máximo 2 documentos
        salida.append(f"[{tema.upper()}]\n{contenido.strip()}")
    
    return "\n\n".join(salida)


schema_buscar = {
    "name": "buscar_conocimiento",
    "description": "Busca información en la base de conocimiento del curso. Usá esta herramienta SIEMPRE antes de responder preguntas técnicas o sobre el curso.",
    "input_schema": {
        "type": "object",
        "properties": {
            "consulta": {
                "type": "string",
                "description": "La consulta de búsqueda. Usá palabras clave específicas."
            }
        },
        "required": ["consulta"]
    }
}

# Probar la búsqueda directamente
print(buscar_conocimiento("claude api precio instalar"))

## Parte 2 — Agente RAG Básico

El agente busca en la base de conocimiento antes de responder cualquier pregunta técnica.
Fijate la instrucción clave: `SIEMPRE buscá antes de responder`.

In [ ]:
def agente_rag(pregunta: str, historial: list = None) -> tuple:
    """Agente RAG que consulta la base de conocimiento antes de responder."""
    
    if historial is None:
        historial = []
    
    historial.append({"role": "user", "content": pregunta})
    print(f"Estudiante: {pregunta}")
    print("-" * 60)
    
    system = """Sos el asistente de soporte del curso 'Pol Desaf-IA' sobre agentes de IA con Claude.
    
    Reglas importantes:
    1. SIEMPRE buscá en la base de conocimiento antes de responder preguntas técnicas
    2. Si la búsqueda no devuelve resultados, decilo claramente
    3. Basá tus respuestas SOLO en la información encontrada, no en tu conocimiento general
    4. Si el estudiante pregunta algo que no está en la base de conocimiento, sugerile que lo pregunte en la comunidad
    5. Respondé en español, de forma amigable y clara"""
    
    while True:
        respuesta = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=1024,
            system=system,
            tools=[schema_buscar],
            messages=historial
        )
        
        if respuesta.stop_reason == "tool_use":
            uso = next(b for b in respuesta.content if b.type == "tool_use")
            print(f"  [Buscando: '{uso.input['consulta']}']")
            resultado = buscar_conocimiento(**uso.input)
            historial.append({"role": "assistant", "content": respuesta.content})
            historial.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": uso.id, "content": resultado}]})
            continue
        
        texto = next(b.text for b in respuesta.content if b.type == "text")
        historial.append({"role": "assistant", "content": texto})
        print(f"Asistente: {texto}")
        return texto, historial


# Prueba
historial = []
_, historial = agente_rag("¿Cómo instalo la API de Claude en Python?", historial)

In [ ]:
print("\n" + "="*60)
# Segunda pregunta — el agente recuerda el contexto
_, historial = agente_rag("¿Y qué diferencia hay con Claude Code?", historial)

## Parte 3 — Patrón Maker-Checker

Dos agentes trabajan juntos para dar respuestas más precisas:
- **Maker:** busca y genera una respuesta inicial
- **Checker:** verifica esa respuesta buscando nuevamente y la refina si hace falta

In [ ]:
def agente_maker(pregunta: str) -> str:
    """Genera una respuesta inicial basada en la base de conocimiento."""
    mensajes = [{"role": "user", "content": pregunta}]
    system = """Sos el Maker: tu trabajo es buscar información en la base de conocimiento
    y generar una respuesta completa. Buscá una o dos veces para asegurarte de tener
    toda la información relevante. Sé detallado en tu respuesta."""
    
    while True:
        respuesta = client.messages.create(
            model="claude-opus-4-5", max_tokens=800, system=system,
            tools=[schema_buscar], messages=mensajes
        )
        if respuesta.stop_reason == "tool_use":
            uso = next(b for b in respuesta.content if b.type == "tool_use")
            resultado = buscar_conocimiento(**uso.input)
            mensajes.append({"role": "assistant", "content": respuesta.content})
            mensajes.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": uso.id, "content": resultado}]})
            continue
        return next(b.text for b in respuesta.content if b.type == "text")


def agente_checker(pregunta: str, respuesta_maker: str) -> str:
    """Verifica y refina la respuesta del Maker."""
    prompt = f"""Pregunta original: {pregunta}

Respuesta del Maker:
{respuesta_maker}

Tu trabajo: verificá esta respuesta buscando en la base de conocimiento.
Si la respuesta es correcta y completa, confirmala con pequeñas mejoras.
Si falta información importante, agregala. Si hay algo incorrecto, corregilo."""
    
    mensajes = [{"role": "user", "content": prompt}]
    system = """Sos el Checker: verificás y refinás respuestas. Buscá en la base de conocimiento
    para validar la información. Devolvé la respuesta final mejorada."""
    
    while True:
        respuesta = client.messages.create(
            model="claude-opus-4-5", max_tokens=800, system=system,
            tools=[schema_buscar], messages=mensajes
        )
        if respuesta.stop_reason == "tool_use":
            uso = next(b for b in respuesta.content if b.type == "tool_use")
            resultado = buscar_conocimiento(**uso.input)
            mensajes.append({"role": "assistant", "content": respuesta.content})
            mensajes.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": uso.id, "content": resultado}]})
            continue
        return next(b.text for b in respuesta.content if b.type == "text")


# Demo del patrón Maker-Checker
pregunta = "¿Qué es MCP y cómo se relaciona con los agentes de Claude?"

print(f"Pregunta: {pregunta}")
print("\n--- MAKER generando respuesta... ---")
respuesta_inicial = agente_maker(pregunta)
print(f"Maker: {respuesta_inicial}")

print("\n--- CHECKER verificando... ---")
respuesta_final = agente_checker(pregunta, respuesta_inicial)
print(f"Checker (respuesta final): {respuesta_final}")

## Parte 4 — Memoria de Usuario

Podemos agregar un perfil de usuario que se actualiza con cada conversación,
permitiendo al agente personalizar las respuestas.

In [ ]:
# Perfil de usuario — simula memoria persistente
perfil_usuario = {
    "nombre": "Pablo",
    "nivel": "principiante",
    "temas_vistos": [],
    "dudas_frecuentes": []
}

def agente_con_memoria(pregunta: str, perfil: dict, historial: list = None) -> tuple:
    """Agente RAG que personaliza respuestas según el perfil del usuario."""
    if historial is None:
        historial = []
    
    historial.append({"role": "user", "content": pregunta})
    
    system = f"""Sos el asistente del curso Pol Desaf-IA.

Perfil del estudiante:
- Nombre: {perfil['nombre']}
- Nivel: {perfil['nivel']}
- Temas ya vistos: {', '.join(perfil['temas_vistos']) if perfil['temas_vistos'] else 'ninguno todavía'}

Reglas:
1. SIEMPRE buscá en la base de conocimiento primero
2. Adaptá la complejidad de tu respuesta al nivel del estudiante
3. Si mencionás un tema nuevo, marcalo como visto
4. Usá el nombre del estudiante para personalizar"""
    
    while True:
        respuesta = client.messages.create(
            model="claude-opus-4-5", max_tokens=800, system=system,
            tools=[schema_buscar], messages=historial
        )
        if respuesta.stop_reason == "tool_use":
            uso = next(b for b in respuesta.content if b.type == "tool_use")
            resultado = buscar_conocimiento(**uso.input)
            # Actualizar temas vistos basado en la búsqueda
            tema_buscado = uso.input["consulta"].split()[0]
            if tema_buscado not in perfil["temas_vistos"]:
                perfil["temas_vistos"].append(tema_buscado)
            historial.append({"role": "assistant", "content": respuesta.content})
            historial.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": uso.id, "content": resultado}]})
            continue
        
        texto = next(b.text for b in respuesta.content if b.type == "text")
        historial.append({"role": "assistant", "content": texto})
        return texto, historial, perfil


historial_m = []
respuesta, historial_m, perfil_usuario = agente_con_memoria(
    "¿Qué necesito saber de Python para empezar el curso?",
    perfil_usuario, historial_m
)
print(f"Agente: {respuesta}")
print(f"\nTemas registrados: {perfil_usuario['temas_vistos']}")

## Resumen

| Concepto | Lo que construiste |
|---|---|
| Base de conocimiento | Documentos propios como herramienta del agente |
| RAG básico | Agente que busca antes de responder |
| Maker-Checker | Doble verificación para respuestas más precisas |
| Memoria de usuario | Perfil que se actualiza con cada conversación |

En producción, reemplazarías el diccionario por una búsqueda vectorial real (ej. con `chromadb` o `pinecone`) para manejar miles de documentos.

---
En la **Lección 06** vamos a ver cómo hacer que estos agentes sean seguros y confiables.